In [1]:
from pathlib import Path
from hydra import initialize, compose
from repo_overview import ParquetDataProcessor
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [22]:
def balanced_weed_sampling(group):
    # Desired sample size in terms of number of images for each common name
    # This is a simplified example; adjust as needed
    desired_images_per_group = 200
    desired_cutouts_per_group = 1000 # Example target, adjust as needed
    # Sort by cutout_count to increase chances of balanced cutout_counts
    group = group.sort_values(by='cutout_id', ascending=False)
    sampled_rows = []
    imgs = []
    images_sampled = 0
    cutouts_sampled = 0
    for _, row in group.iterrows():
        if images_sampled < desired_images_per_group and cutouts_sampled < desired_cutouts_per_group:
            sampled_rows.append(row)
            
            img = row["image_id"]
            cut = row["cutout_id"]
            if img not in imgs:
                imgs.append(img)
                images_sampled += 1
            cutouts_sampled += cut
    return pd.DataFrame(sampled_rows)

In [2]:
with initialize(version_base="1.3", config_path="../conf"):
    cfg = compose(config_name="config.yaml", return_hydra_config=True)
    cfg.general.batch_id = "repo_overview"
    cfg.hydra.runtime.cwd = "/home/psa_images/SemiF-AnnotationPipeline"
    cfg.data.database_parquet = (
        "/home/psa_images/SemiF-AnnotationPipeline/repo_overview/database"
    )
    cfg.data.repo_database = "/home/psa_images/SemiF-AnnotationPipeline/repo_overview"

In [3]:
processor = ParquetDataProcessor(cfg)
processor.read_parquet_file(totals=False)

In [5]:
df = processor.df.copy()

In [6]:
mdf = df[["image_id","batch_id", "cutout_id", "common_name", "general_season"]]
mdf = mdf[mdf["common_name"]!= "Colorchecker"]
mdf = mdf[mdf["common_name"]!= "Unknown"]


In [ ]:
mdf

In [25]:
cutout_groupby = mdf.groupby(["common_name","image_id"])["cutout_id"].nunique().reset_index().sort_values(by="cutout_id", ascending=False)

exclude_species = ["Jungle rice", "Spiny amaranth", "Common sunflower", "Ragweed parthenium", "Kochia"]
cutouts_excluded = cutout_groupby[~cutout_groupby["common_name"].isin(exclude_species)]
# cutouts_excluded = cutouts_excluded[cutouts_excluded["cutout_id"]> ]

sampled = cutouts_excluded.drop_duplicates(subset="image_id").sample(frac=0.1, random_state=42)

# equal probability weighting
print(sampled.shape)
rep_dataset = df[df["image_id"].isin(sampled["image_id"])].drop_duplicates(subset="image_id")
rep_dataset.to_csv("representative_dataset_6326.csv", index=False)
# sampled.groupby(["common_name"])["cutout_id"].sum().reset_index().sort_values(by="cutout_id")
# sampled.to_csv("balanced_dataset.csv", index=False)


(6326, 3)
